# Notebook 3 — Train Honest vs Leaky

This is your headline experiment (RQ2). The **same model** is trained two ways:

- **Honest** folds (patient-disjoint) — the correct number
- **Leaky** folds (random slice) — the inflated number many papers report

You get two findings:
1. **Accuracy gap** — leaky scores higher
2. **Variance collapse** — leaky folds look artificially consistent, hiding that some patients are genuinely hard. *This is the stronger result.*

It also saves each trained model and its predictions.

In [1]:
# Cell 1 — imports
!pip install timm --quiet

from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import timm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| timm", timm.__version__)
if device == "cpu":
    print("WARNING: no GPU. Turn on the T4 accelerator or this will be very slow.")

device: cuda | timm 1.0.26


## Cell 2 — paths and settings

`ARCH` is the model. `resnet18` is fast and was used in the pilot. `FOLDS` lets you run one fold first (`[0]`) before committing to all five.

In [2]:
# Cell 2 
IN_DIR   = "/kaggle/input"          
OUT_DIR  = "/kaggle/working/runs"  

ARCH       = "resnet18"
FOLDS      = [0,1,2,3,4]                  # start with [0]; later use [0,1,2,3,4]
EPOCHS     = 20
BATCH_SIZE = 32
LR         = 3e-4
SEED       = 42

In [3]:
# Cell 3 — locate inputs 
def find(name):
    hits = list(Path(IN_DIR).rglob(name))
    assert hits, f"{name} not found under {IN_DIR}. Add the right dataset."
    return hits[0]

images_path   = find("images.npy")
manifest_path = find("manifest_splits.csv")
print("images  :", images_path)
print("manifest:", manifest_path)

images   = np.load(images_path)
manifest = pd.read_csv(manifest_path)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("images", images.shape, "| slices", len(manifest))

images  : /kaggle/input/notebooks/souravpaul100304/1-cheng-dataset-analysis/data/images.npy
manifest: /kaggle/input/notebooks/souravpaul100304/2-make-splits/data/manifest_splits.csv
images (3064, 224, 224) | slices 3064


## Cell 4 — dataset

Each grayscale slice becomes a 3-channel tensor (pretrained nets expect RGB) and is normalised with ImageNet statistics.

**Augmentation is deliberately mild** — just a horizontal flip. Heavy augmentation would blur the honest-vs-leaky comparison, which is the exact thing being measured.

In [4]:
# Cell 4 — dataset
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class SliceDataset(Dataset):
    def __init__(self, images, labels, train=False):
        self.images, self.labels, self.train = images, labels, train
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        img = self.images[i].astype(np.float32) / 255.0
        if self.train and np.random.rand() < 0.5:
            img = np.fliplr(img).copy()
        img = np.stack([img]*3, axis=-1)
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(img.transpose(2,0,1)), int(self.labels[i])

print("dataset defined")

dataset defined


## Cell 5 — train / eval helpers

In [5]:
# Cell 5 — helpers
def run_epoch(model, loader, criterion, optimizer, train):
    model.train() if train else model.eval()
    total, probs_all, true_all = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(y)
            probs_all.append(torch.softmax(logits,1).detach().cpu().numpy())
            true_all.append(y.cpu().numpy())
    probs = np.concatenate(probs_all); true = np.concatenate(true_all)
    return total/len(true), probs, true

def score(true, probs):
    pred = probs.argmax(1)
    return {"accuracy": accuracy_score(true, pred),
            "macro_f1": f1_score(true, pred, average="macro"),
            "auc": roc_auc_score(true, probs, multi_class="ovr", average="macro")}
print("helpers defined")

helpers defined


## Cell 6 — train one model

Trains one architecture on one fold of one split, then saves the weights and the test-set predictions. 

In [6]:
# Cell 6 — train-one function
def train_one(split_col, fold):
    test_mask = (manifest[split_col] == fold).values
    tr_idx = np.where(~test_mask)[0]
    te_idx = np.where(test_mask)[0]
    labels = manifest["label"].values - 1     # 1,2,3 -> 0,1,2

    tr_dl = DataLoader(SliceDataset(images[tr_idx], labels[tr_idx], True),
                       batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    te_dl = DataLoader(SliceDataset(images[te_idx], labels[te_idx], False),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    torch.manual_seed(SEED)
    model = timm.create_model(ARCH, pretrained=True, num_classes=3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for ep in range(EPOCHS):
        tr_loss, _, _ = run_epoch(model, tr_dl, criterion, optimizer, True)
        scheduler.step()
        if (ep+1) % 5 == 0:
            _, p, t = run_epoch(model, te_dl, criterion, None, False)
            print(f"    epoch {ep+1:2d}  loss {tr_loss:.3f}  acc {score(t,p)['accuracy']:.4f}")

    _, probs, true = run_epoch(model, te_dl, criterion, None, False)
    m = score(true, probs)

    tag = f"{ARCH}_{split_col}_fold{fold}"
    torch.save(model.state_dict(), Path(OUT_DIR)/f"model_{tag}.pt")
    np.savez(Path(OUT_DIR)/f"preds_{tag}.npz", probs=probs, true=true, test_idx=te_idx)
    return m
print("train_one defined")

train_one defined


## Cell 7 — run

Loops over both splits and the folds you chose.

In [7]:
# Cell 7 — run
results = []
for split_col in ["fold_honest", "fold_leaky"]:
    for fold in FOLDS:
        print(f">>> {split_col} fold {fold}")
        m = train_one(split_col, fold)
        print(f"    FINAL acc {m['accuracy']:.4f}  f1 {m['macro_f1']:.4f}  auc {m['auc']:.4f}")
        results.append({"arch": ARCH, "split": split_col, "fold": fold, **m})

results_df = pd.DataFrame(results)
results_df.to_csv(Path(OUT_DIR)/"results.csv", index=False)
print("\nsaved results.csv and model/pred files to", OUT_DIR)

>>> fold_honest fold 0


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

    epoch  5  loss 0.046  acc 0.9395
    epoch 10  loss 0.011  acc 0.9519
    epoch 15  loss 0.003  acc 0.9504
    epoch 20  loss 0.002  acc 0.9550
    FINAL acc 0.9550  f1 0.9518  auc 0.9957
>>> fold_honest fold 1
    epoch  5  loss 0.042  acc 0.8563
    epoch 10  loss 0.012  acc 0.8862
    epoch 15  loss 0.005  acc 0.8899
    epoch 20  loss 0.003  acc 0.8881
    FINAL acc 0.8881  f1 0.8777  auc 0.9660
>>> fold_honest fold 2
    epoch  5  loss 0.049  acc 0.9684
    epoch 10  loss 0.013  acc 0.9734
    epoch 15  loss 0.004  acc 0.9817
    epoch 20  loss 0.003  acc 0.9817
    FINAL acc 0.9817  f1 0.9805  auc 0.9990
>>> fold_honest fold 3
    epoch  5  loss 0.036  acc 0.9182
    epoch 10  loss 0.006  acc 0.9168
    epoch 15  loss 0.003  acc 0.9153
    epoch 20  loss 0.002  acc 0.9270
    FINAL acc 0.9270  f1 0.9117  auc 0.9850
>>> fold_honest fold 4
    epoch  5  loss 0.034  acc 0.9413
    epoch 10  loss 0.009  acc 0.9379
    epoch 15  loss 0.007  acc 0.9413
    epoch 20  loss 0.004  acc

## Cell 8 — the two findings

**Finding 1 — accuracy gap.** Leaky mean minus honest mean.

**Finding 2 — variance collapse.** Compare the standard deviations. Leaky std much smaller than honest std means leakage manufactures false confidence: it hides that some patients are hard. Report this as prominently as the mean gap — arguably more.

*(Full effect needs all 5 folds. One fold can't show variance.)*

In [8]:
# Cell 8 — summarise
summary = results_df.groupby("split")[["accuracy","macro_f1","auc"]].agg(["mean","std"])
print(summary.round(4).to_string())

if len(FOLDS) > 1:
    h = results_df[results_df.split=="fold_honest"].accuracy
    l = results_df[results_df.split=="fold_leaky"].accuracy
    print(f"\nAccuracy inflation (leaky - honest): {(l.mean()-h.mean())*100:+.2f} pts")
    if h.std() > 0:
        print(f"Variance ratio (honest std / leaky std): {h.std()/l.std():.1f}x")
        print("  -> leaky folds look artificially consistent; that is the false-confidence effect")
else:
    print("\nRun all 5 folds (FOLDS=[0,1,2,3,4]) to see the variance-collapse effect.")

            accuracy         macro_f1             auc        
                mean     std     mean     std    mean     std
split                                                        
fold_honest   0.9373  0.0347   0.9303  0.0390  0.9865  0.0129
fold_leaky    0.9794  0.0034   0.9769  0.0041  0.9985  0.0008

Accuracy inflation (leaky - honest): +4.22 pts
Variance ratio (honest std / leaky std): 10.2x
  -> leaky folds look artificially consistent; that is the false-confidence effect
